# Phase 3 RQ1 v2 — B0 training

**This is the only stage that requires a Kaggle GPU/CUDA runtime.** Turn Internet on, attach the private dataset containing `phase3_rq1_v2_inputs.zip`, accept the gated `meta-llama/Llama-3.2-1B` terms, and add a Kaggle secret named `HF_TOKEN`.

Run this notebook once for each seed: set `SEED_TO_RUN` to `42`, then `43`, then `44`. Each run trains B0 over all five frozen stream orders and checks that its selected XNLI data hash matches the input-preparation bundle before training. Download each `phase3_rq1_v2_b0_seed_<seed>.zip` and keep all three. The ZIP contains result records and provenance, not checkpoints or XNLI text.

Three seeds are the requested initial v2 pilot; they do not guarantee that any prior will be statistically supported.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import uuid

REPO_URL = "https://github.com/Mani-S-2709/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
elif (REPO_ROOT / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
else:
    raise FileExistsError(f"Repository path exists but is not a Git clone: {REPO_ROOT}")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    str(REPO_ROOT / "phase2" / "requirements-phase2.txt"),
], check=True)

from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Kaggle secret is empty.")
os.environ["HF_TOKEN"] = hf_token
del hf_token

sys.path.insert(0, str(REPO_ROOT))
input_bundles = list(Path("/kaggle/input").rglob("phase3_rq1_v2_inputs.zip"))
if len(input_bundles) != 1:
    raise FileNotFoundError(
        "Attach exactly one private Kaggle dataset containing "
"phase3_rq1_v2_inputs.zip. Found: " + str(input_bundles)
    )

SEED_TO_RUN = 42  # Change to 43 and 44 for the remaining two runs.
if SEED_TO_RUN not in (42, 43, 44):
    raise ValueError("SEED_TO_RUN must be 42, 43, or 44.")

OUTPUT_DIR = Path("/kaggle/working/phase3_rq1_v2_seed_outputs")
WORK_DIR = Path("/kaggle/temp/phase3_rq1_v2") / f"seed_{SEED_TO_RUN}_{uuid.uuid4().hex}"
subprocess.run([
    sys.executable, "-m", "phase3.rq1_v2_workflow", "train-seed",
    "--inputs-zip", str(input_bundles[0]),
    "--seed", str(SEED_TO_RUN),
    "--work-dir", str(WORK_DIR),
    "--output-dir", str(OUTPUT_DIR),
], cwd=REPO_ROOT, check=True)

RESULT_ZIP = OUTPUT_DIR / f"phase3_rq1_v2_b0_seed_{SEED_TO_RUN}.zip"
if not RESULT_ZIP.is_file():
    raise FileNotFoundError(f"Validated result bundle was not created: {RESULT_ZIP}")
print(f"Download this seed's result bundle from Kaggle output files: {RESULT_ZIP}")